# Flickr30k T3→S3 Final Method Lock

Khóa `Direct Logit-KD` sau khi ba seed đều tăng CIDEr-D và paired bootstrap
tổng hợp có khoảng tin cậy 95% nằm hoàn toàn trên 0. Seed có Validation
CIDEr-D cao nhất chỉ được chọn làm checkpoint triển khai; kết luận khoa học
vẫn dựa trên trung bình và độ bất định ba seed.

## Kaggle input bắt buộc — đúng 4 input

1. Output `F30K_T3S3_LogitKD_Aggregate`.
2. Output `F30K_T3S3_LogitKD_42`.
3. Output `F30K_T3S3_LogitKD_123`.
4. Output `F30K_T3S3_LogitKD_2026`.

Runtime: CPU/Accelerator None, Internet Off, dự kiến dưới 2 phút.
Không gắn dữ liệu ảnh, Teacher, baseline hoặc Held-out Test. Notebook này
không mở Test; Test tiếp tục bị khóa trong khi chờ T1→S1 DenseMap aggregate.

## Cell 1 — Verify aggregate and three candidate checkpoints

In [1]:
EXPERIMENT_ID = "F30K_T3S3_Final_Lock"


import csv
import hashlib
import json
import math
import os
import statistics
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

def verify_manifest(root):
    manifest_path = Path(root) / "MANIFEST.json"
    manifest = read_json(manifest_path)
    assert manifest["status"] == "complete"
    assert manifest["test_set_status"] == "sealed"
    assert manifest["test_set_used"] is False
    for artifact in manifest["artifacts"]:
        path = Path(root) / artifact["relative_path"]
        assert path.is_file(), path
        assert path.stat().st_size == int(artifact["size_bytes"])
        assert sha256(path) == artifact["sha256"]
    return manifest_path, manifest

AGGREGATE_ID = "F30K_T3S3_LogitKD_Aggregate"
aggregate_path = unique_named(AGGREGATE_ID + "_RESULT.json")
aggregate = read_json(aggregate_path)
aggregate_manifest_path, _ = verify_manifest(aggregate_path.parent)

assert aggregate["status"] == "complete"
assert aggregate["experiment_id"] == AGGREGATE_ID
assert aggregate["pair_id"] == "T3S3_LogitKD"
assert aggregate["method"] == "Direct Logit-KD"
assert aggregate["seeds"] == [42, 123, 2026]
assert aggregate["run_count"] == 3
assert aggregate["consistent_positive_delta"] is True
assert aggregate["consistent_negative_delta"] is False
assert aggregate["decision"] == "PROMOTE_TO_FINAL_METHOD_LOCK"
assert aggregate["test_set_status"] == "sealed"
assert aggregate["test_set_used"] is False

candidates = []
for seed in (42, 123, 2026):
    experiment = f"F30K_T3S3_LogitKD_{seed}"
    result_path = unique_named(experiment + "_RESULT.json")
    result = read_json(result_path)
    result_manifest_path, _ = verify_manifest(result_path.parent)
    assert result["status"] == "complete"
    assert result["experiment_id"] == experiment
    assert result["teacher_id"] == "T3"
    assert result["student_id"] == "S3"
    assert result["method"] == "Direct Logit-KD"
    assert int(result["seed"]) == seed
    assert result["test_set_status"] == "sealed"
    assert result["test_set_used"] is False
    best_path = result_path.parent / "best.pt"
    assert sha256(best_path) == result["artifacts"]["best.pt"]["sha256"]

    aggregate_run = next(
        row for row in aggregate["runs"] if int(row["seed"]) == seed
    )
    metrics = result["best_validation_metrics"]
    assert math.isclose(
        float(metrics["CIDEr_D"]),
        float(aggregate_run["kd_CIDEr_D"]),
        rel_tol=0.0,
        abs_tol=1e-12,
    )
    candidates.append({
        "seed": seed,
        "experiment_id": experiment,
        "best_epoch": int(result["best_epoch"]),
        "validation_BLEU_1": float(metrics["BLEU_1"]),
        "validation_BLEU_4": float(metrics["BLEU_4"]),
        "validation_METEOR": float(metrics["METEOR"]),
        "validation_ROUGE_L": float(metrics["ROUGE_L"]),
        "validation_CIDEr_D": float(metrics["CIDEr_D"]),
        "validation_ce_loss": float(metrics["ce_loss"]),
        "delta_CIDEr_D_vs_same_seed_baseline": float(
            result["delta_validation_cider_d"]
        ),
        "best_checkpoint_sha256": sha256(best_path),
        "best_checkpoint_bytes": best_path.stat().st_size,
        "result_sha256": sha256(result_path),
        "manifest_sha256": sha256(result_manifest_path),
    })

# Deployment checkpoint only: the scientific estimate remains the three-seed
# mean and uncertainty. Tie-break order is fixed before reading Test.
selected = max(
    candidates,
    key=lambda row: (
        row["validation_CIDEr_D"],
        row["validation_BLEU_4"],
        -row["validation_ce_loss"],
        -row["best_epoch"],
    ),
)
assert selected["seed"] == 42

combined = aggregate["combined_paired_bootstrap"]
assert float(combined["ci95_lower"]) > 0
assert float(combined["bootstrap_positive_probability"]) == 1.0

lock = {
    "schema_version": "f30k_t3s3_final_method_lock_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "locked",
    "experiment_id": EXPERIMENT_ID,
    "pair": "T3 SigLIP ViT-B/16 adapted -> S3 TinyViT-5M",
    "method": "Direct Logit-KD",
    "scientific_result": {
        "seed_count": 3,
        "validation_CIDEr_D_delta_mean": aggregate["summary"]["CIDEr_D"][
            "delta_mean"
        ],
        "validation_CIDEr_D_delta_sample_sd": aggregate["summary"]["CIDEr_D"][
            "delta_sample_sd"
        ],
        "combined_paired_bootstrap": combined,
        "decision": aggregate["decision"],
    },
    "deployment_checkpoint_selection": {
        "role": "single checkpoint selected by Validation only",
        "tie_break_order": [
            "higher CIDEr-D", "higher BLEU-4",
            "lower CE", "earlier epoch",
        ],
        **selected,
    },
    "all_candidate_checkpoints": candidates,
    "aggregate_result_sha256": sha256(aggregate_path),
    "aggregate_manifest_sha256": sha256(aggregate_manifest_path),
    "method_or_threshold_changes_after_lock": False,
    "held_out_test_authorized_now": False,
    "held_out_test_prerequisite": (
        "Complete T1S1 Dense Mapped Logit-KD three-seed aggregate and "
        "create the global Validation-only registry."
    ),
    "next_phase_authorized": (
        "WAIT_T1S1_DENSEMAP_AGGREGATE_THEN_F30K_GLOBAL_VALIDATION_LOCK"
    ),
    "test_set_status": "sealed",
    "test_set_used": False,
}

lock_path = WORK_ROOT / "F30K_T3S3_FINAL_METHOD_LOCKED.json"
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
csv_path = WORK_ROOT / "candidate_checkpoints.csv"
report_path = WORK_ROOT / "REPORT.md"
write_json(lock_path, lock)
write_json(result_path, lock)
with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(candidates[0]))
    writer.writeheader()
    writer.writerows(candidates)
report_path.write_text(
    "# Flickr30k T3→S3 final method lock\n\n"
    f"- Method: `Direct Logit-KD`\n"
    f"- Mean Validation CIDEr-D delta: "
    f"`{aggregate['summary']['CIDEr_D']['delta_mean']:+.6f}`\n"
    f"- Combined paired-bootstrap CI95: "
    f"`[{combined['ci95_lower']:+.6f}; {combined['ci95_upper']:+.6f}]`\n"
    f"- Deployment checkpoint: seed `{selected['seed']}` "
    f"(CIDEr-D `{selected['validation_CIDEr_D']:.6f}`)\n"
    "- Held-out Test: `SEALED / NOT USED`\n"
    "- Next: wait for T1→S1 DenseMap aggregate, then global Validation lock.\n",
    encoding="utf-8",
)

artifacts = []
for path in [lock_path, result_path, csv_path, report_path]:
    artifacts.append({
        "relative_path": path.name,
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })
write_json(WORK_ROOT / "MANIFEST.json", {
    "schema_version": "f30k_t3s3_final_lock_manifest_v1",
    "experiment_id": EXPERIMENT_ID,
    "status": "complete",
    "artifacts": artifacts,
    "test_set_status": "sealed",
    "test_set_used": False,
})

print("Status                 : LOCKED")
print("Method                 : Direct Logit-KD")
print("Three-seed mean delta  :",
      aggregate["summary"]["CIDEr_D"]["delta_mean"])
print("Combined CI95          :", combined["ci95_lower"],
      combined["ci95_upper"])
print("Deployment seed        :", selected["seed"])
print("Deployment CIDEr-D     :", selected["validation_CIDEr_D"])
print("Held-out Test          : SEALED / NOT AUTHORIZED YET")
print("Next                   :", lock["next_phase_authorized"])

Status                 : LOCKED
Method                 : Direct Logit-KD
Three-seed mean delta  : 0.03389222163311526
Combined CI95          : 0.019477099305418074 0.04817347479190771
Deployment seed        : 42
Deployment CIDEr-D     : 0.617055344048548
Held-out Test          : SEALED / NOT AUTHORIZED YET
Next                   : WAIT_T1S1_DENSEMAP_AGGREGATE_THEN_F30K_GLOBAL_VALIDATION_LOCK
